# Integração Deepseek — Intenção `ingredientes`

Este notebook adiciona uma intenção `ingredientes` ao projeto e implementa um fluxo interativo que: detecta a intenção, pede ao usuário qual lanche deseja, valida a escolha e consulta a API Deepseek para obter a lista de ingredientes/receita.

Os pratos (lanches) usados neste exemplo são os do menu: Hambúrguer, X-Burger, X-Salada, X-Bacon, X-Egg, X-Calabresa, X-Frango, X-Tudo e Vegetariano.

## README rápido — como executar

1. Instale as dependências (se ainda não instalado):

```bash
pip install -r requirements.txt
# ou apenas requests se já tiver outras dependências
pip install requests
```

2. Defina sua API key da Deepseek como variável de ambiente `DEEPSEEK_API_KEY`. Exemplos:

- No PowerShell (temporário nesta sessão):
```powershell
$env:DEEPSEEK_API_KEY = "SUA_API_KEY_AQUI"
```

- Para tornar permanente no Windows (setx) — **não recomendado em notebooks**:
```powershell
setx DEEPSEEK_API_KEY "SUA_API_KEY_AQUI"
```

3. Execute as células do notebook em ordem. Quando a intenção `ingredientes` for detectada, você será solicitado a escolher o(s) lanche(s) e fornecer a API key (se não estiver definida).

**Atenção:** Não coloque a API key no código do notebook. Use variável de ambiente ou entrada interativa conforme instruções.

In [5]:
import os
import requests
import json
from typing import List, Tuple

# Pratos (lanches) do menu
PRATOS = [
    "Hambúrguer",
    "X-Burger",
    "X-Salada",
    "X-Bacon",
    "X-Egg",
    "X-Calabresa",
    "X-Frango",
    "X-Tudo",
    "Vegetariano"
]

INTENT_INGREDIENTES = [
    "ingredientes",
    "quais são os ingredientes",
    "o que tem no prato",
    "buscar ingredientes",
    "preciso dos ingredientes",
    "receita do lanche",
    "receita do prato",
    "o que tem no lanche"
]

def detect_intent(text: str) -> str:
    txt = (text or '').lower()
    for kw in INTENT_INGREDIENTES:
        if kw in txt:
            return 'ingredientes'
    return 'outro'

# Teste rápido
if __name__ == '__main__':
    print('Lista de pratos disponíveis:')
    print(PRATOS)

Lista de pratos disponíveis:
['Hambúrguer', 'X-Burger', 'X-Salada', 'X-Bacon', 'X-Egg', 'X-Calabresa', 'X-Frango', 'X-Tudo', 'Vegetariano']


In [9]:
def escolha_prato_interativa(pratos: List[str] = PRATOS) -> List[str]:
    """Exibe as opções e pede seleção ao usuário.
    Aceita números (ex: 1 ou 1,2) ou nomes (ex: X-Burger).
    Retorna lista de nomes válidos.
    """
    print('Pratos disponíveis:')
    for i, p in enumerate(pratos, start=1):
        print(f'{i}. {p}')

    escolha = input('Escolha o número do prato (ou nomes separados por vírgula): ').strip()
    # tenta parse por índices
    selecionados: List[str] = []
    if not escolha:
        print('Nenhuma seleção fornecida. Tente novamente.')
        return escolha_prato_interativa(pratos)

    # se só contém dígitos/virgulas, interpretar como índices
    parts = [s.strip() for s in escolha.split(',') if s.strip()]
    # Detecta se a maioria das partes é numérica
    num_count = sum(1 for p in parts if p.isdigit())
    if num_count >= 1:
        for p in parts:
            if p.isdigit():
                idx = int(p) - 1
                if 0 <= idx < len(pratos):
                    selecionados.append(pratos[idx])
    else:
        # Interpretar como nomes ou partes de nomes
        for p in parts:
            found = False
            for prato in pratos:
                if p.lower() == prato.lower() or p.lower() in prato.lower():
                    selecionados.append(prato)
                    found = True
                    break
            if not found:
                print(f'Opção inválida: {p}')

    if not selecionados:
        print('Seleção inválida. Tente novamente.')
        return escolha_prato_interativa(pratos)

    # remove duplicatas mantendo ordem
    seen = set()
    unique = []
    for s in selecionados:
        if s not in seen:
            unique.append(s)
            seen.add(s)
    return unique

# Teste interativo (não executa automaticamente nas células de import)
if __name__ == '__main__':
    escolha = escolha_prato_interativa()
    print('Selecionados:', escolha)

Pratos disponíveis:
1. Hambúrguer
2. X-Burger
3. X-Salada
4. X-Bacon
5. X-Egg
6. X-Calabresa
7. X-Frango
8. X-Tudo
9. Vegetariano
Selecionados: ['Hambúrguer']
Selecionados: ['Hambúrguer']


In [ ]:
import requests
import json

def consulta_deepseek(nome_prato: str, api_key: str, timeout: int = 20) -> str:
    """Faz uma chamada à API Deepseek (via OpenRouter endpoint) para pedir a receita/ingredientes do prato.
    O prompt foi atualizado para usar formato de agent com JSON estruturado e informações sobre alergênicos.
    """
    url = 'https://openrouter.ai/api/v1/chat/completions'
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
    }
    # Prompt atualizado com contextualização de agent e formato JSON estruturado
    prompt = f"""# Contextualização

Você é um chef de cozinha da região metropolitana de São Paulo Brasil e trabalha numa hamburgueria renomada. Seu restaurante tem ótimas recomendações.

# Tarefa

Passe os ingredientes da receita do {nome_prato} que você faz, incluindo informações sobre os possíveis alergênicos.
Traga informações única e exclusivamente dos ingredientes, sem modo de preparo e outras informações.

# Formato da resposta

Responda no seguinte formato JSON:

{{
  "ingredientes": [
    {{
      "nome": "nome do ingrediente",
      "quantidade": "quantidade do ingrediente",
      "unidade": "unidade de medida",
      "alergenico": true/false
    }}
  ]
}}

# Exemplo de resposta

{{
    "ingredientes": [
        {{
            "nome": "Farinha de Trigo",
            "quantidade": "500",
            "unidade": "gramas",
            "alergenico": true
        }},
        {{
            "nome": "Ovos",
            "quantidade": "3",
            "unidade": "unidades",
            "alergenico": true
        }},
        {{
            "nome": "Açúcar",
            "quantidade": "1",
            "unidade": "xícara",
            "alergenico": false
        }}
    ]
}}"""
    payload = {
        'model': 'deepseek/deepseek-r1:free',
        'messages': [
            {'role': 'user', 'content': prompt}
        ]
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
    except Exception as e:
        return f'Erro na requisição: {e}'

    try:
        j = resp.json()
    except Exception:
        return f'Erro ao decodificar resposta JSON: {resp.text}'

    # Tenta extrair conteúdo esperado (ajuste conforme o formato real da API)
    content = None
    # OpenRouter/Chat completions suelen devolver 'choices' -> [{'message':{'content': '...'}}]
    try:
        content = j.get('choices', [{}])[0].get('message', {}).get('content')
    except Exception:
        content = None

    if not content:
        # fallback: return raw json prettified
        return json.dumps(j, ensure_ascii=False, indent=2)

    return content


In [1]:
# Fluxo exemplo: detecta intenção e, se for 'ingredientes', executa a interação.
def run_interactive_flow():
    user_input = input('Usuário: ').strip()
    intent = detect_intent(user_input)
    if intent != 'ingredientes':
        print('Intent não detectada como ingredientes. Saindo do fluxo de exemplo.')
        return

    # Pergunta qual prato
    selecionados = escolha_prato_interativa(PRATOS)

    # Obtém API key de variável de ambiente ou pede ao usuário
    api_key = os.getenv('DEEPSEEK_API_KEY')
    if not api_key:
        api_key = input("Insira a API key da Deepseek (não a compartilhe): ").strip()

    for prato in selecionados:
        print(f'\nBuscando ingredientes para: {prato}...')
        resultado = consulta_deepseek(prato, api_key)
        print('--- Resultado ---')
        print(resultado)

# Para executar, rode: run_interactive_flow()

## Observações finais e próximos passos

- Você pode integrar `detect_intent` no pipeline do `app.py` (Streamlit) existente: ao detectar `ingredientes`, exibir um selectbox com os pratos e chamar `consulta_deepseek` quando o usuário confirmar.
- Para produção, adicione caching e limites (rate limiting) para evitar consultas excessivas à API.
- Se quiser, eu adapto o código para a interface Streamlit do `app.py` (botões/expander) e integro o resultado diretamente na UI do chatbot.